# Azimuth Kidney Reference — quick exploration

This notebook demonstrates how to inspect a converted Azimuth Human Kidney reference H5AD.
It is intended for lightweight exploration on a laptop: discover embeddings, metadata, QC, and a few expression overlays.

Instructions:
- Edit the `ref_path` variable in the next cell to point to your `.h5ad` (or run the provided R converter to produce one from `ref.Rds`).
- Run the cells in order; they are designed to be self-contained and robust to missing optional packages (scanpy, annoy).


# Azimuth human_kidney import
This repository now contains an imported copy of the Azimuth `human_kidney` reference under `azimuth_human_kidney/`.

- Use `scripts/convert_to_h5ad.R` to produce an `.h5ad` from a pair of RDS files (`ref` and `fullref`).
- This notebook will prefer any `.h5ad` found under `reference/` when selecting `ref_path`.

In [1]:
# Setup: imports and plotting defaults
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Optional: anndata and scanpy (not mandatory but useful)
try:
    import anndata
    import scanpy as sc
except Exception:
    anndata = None
    sc = None

# set seed for reproducibility
SEED = 42
np.random.seed(SEED)

import warnings
# Suppress noisy FutureWarning coming from legacy_api_wrap library used by some IO backends
warnings.filterwarnings("ignore", message=".*dtype argument is deprecated.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="legacy_api_wrap")

print("Python:", sys.version.splitlines()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", mpl.__version__)
if anndata is not None:
    print("anndata:", anndata.__version__)
if sc is not None:
    print("scanpy:", sc.__version__)

# plotting defaults
plt.rcParams['figure.figsize'] = (6, 5)
plt.rcParams['image.cmap'] = 'viridis'

Python: 3.13.1 (v3.13.1:06714517797, Dec  3 2024, 14:00:22) [Clang 15.0.0 (clang-1500.3.9.4)]
NumPy: 2.3.0
Pandas: 2.3.3
Matplotlib: 3.10.8


In [2]:
# Helper functions: embedding finder, UMI extractor, simple display
import numpy as np
import scipy.sparse
from typing import Optional


def find_embedding(adata) -> np.ndarray:
    """Robustly find a 2D embedding in AnnData (search obsm keys or obs columns).

    Returns
    -------
    np.ndarray : (n_cells, 2)
    """
    if adata is None:
        raise ValueError("adata is None")
    # 1) prefer common obsm keys containing 'umap'
    for k in adata.obsm.keys():
        if 'umap' in k.lower():
            emb = adata.obsm[k]
            if getattr(emb, 'ndim', 2) == 2 and emb.shape[1] >= 2:
                return np.asarray(emb)[:, :2]
    # 2) explicit obs columns
    if 'UMAP_1' in adata.obs.columns and 'UMAP_2' in adata.obs.columns:
        return adata.obs[['UMAP_1', 'UMAP_2']].values
    # 3) fallback: any obsm 2D array
    for k in adata.obsm.keys():
        emb = adata.obsm[k]
        if getattr(emb, 'ndim', 2) == 2 and emb.shape[1] >= 2:
            return np.asarray(emb)[:, :2]
    raise ValueError('No 2D embedding found in adata (checked obsm and obs)')


def compute_umis(adata) -> np.ndarray:
    """Robustly extract or compute UMI/library-size per cell.

    Strategy (ordered):
      1) numeric obs columns with common count keys (e.g., nCount_*, n_counts, total_counts)
      2) obs columns matching regex for nCount-like names
      3) adata.layers['counts'] if present
      4) adata.raw.X if present
      5) sum rows of adata.X (fallback)

    Returns
    -------
    np.ndarray : shape (n_obs,)
    """
    import re

    # 1) explicit well-known columns
    candidates = ['nCount_RNA', 'n_counts', 'total_counts', 'library_size']
    for c in candidates:
        if c in adata.obs.columns:
            try:
                return np.asarray(adata.obs[c].values).astype(float)
            except Exception:
                pass

    # Prefer any nCount_* columns (e.g., nCount_refAssay) if present
    ncount_cols = [c for c in adata.obs.columns if re.match(r'(?i)^nCount_', c)]
    for c in ncount_cols:
        try:
            vals = pd.to_numeric(adata.obs[c], errors='coerce')
            if not np.all(np.isnan(vals)):
                vals = vals.fillna(0.0)
                return np.asarray(vals).astype(float)
        except Exception:
            continue

    # 2) search for obs columns like nCount_* or containing 'ncount'
    for col in adata.obs.columns:
        if re.search(r'(?i)ncount|n_counts|total_counts|library_size', col):
            try:
                vals = pd.to_numeric(adata.obs[col], errors='coerce')
                if not np.all(np.isnan(vals)):
                    vals = vals.fillna(0.0)
                    return np.asarray(vals).astype(float)
            except Exception:
                continue

    # 3) layers (counts)
    if hasattr(adata, 'layers') and 'counts' in adata.layers:
        arr = adata.layers['counts']
        if scipy.sparse.issparse(arr):
            sums = np.asarray(arr.sum(axis=1)).reshape(-1).astype(float)
        else:
            sums = np.sum(arr, axis=1).astype(float)
        if sums.sum() > 0:
            return sums

    # 4) raw.X
    if getattr(adata, 'raw', None) is not None and getattr(adata.raw, 'X', None) is not None:
        rawX = adata.raw.X
        if scipy.sparse.issparse(rawX):
            sums = np.asarray(rawX.sum(axis=1)).reshape(-1).astype(float)
        else:
            sums = np.sum(rawX, axis=1).astype(float)
        if sums.sum() > 0:
            return sums

    # 5) fallback to adata.X
    if scipy.sparse.issparse(adata.X):
        sums = np.asarray(adata.X.sum(axis=1)).reshape(-1).astype(float)
    else:
        sums = np.sum(adata.X, axis=1).astype(float)

    if sums.sum() == 0:
        import warnings
        warnings.warn("All sums in adata.X are zero; check for count columns in adata.obs or counts in layers/raw.")
    return sums


def show_table(df, n=5):
    print(df.head(n))
    print(f"(shape: {df.shape})")

In [3]:
from pathlib import Path

az_hk_ref_dir = Path("reference")
found_h5ads = (
    sorted(list(az_hk_ref_dir.glob("*.h5ad")), key=lambda p: p.name)
    if az_hk_ref_dir.exists()
    else []
)
if len(found_h5ads) > 0:
    pref_order = ["ref_kpmp.h5ad", "ref_lake.h5ad", "ref.h5ad"]
    for pref_name in pref_order:
        pref = [p for p in found_h5ads if p.name == pref_name]
        if len(pref) > 0:
            ref_path = pref[0]
            break
    else:
        ref_path = found_h5ads[0]
else:
    ref_path = Path("reference/ref_kpmp.h5ad")

# optional Annoy index next to the h5ad or in the human_kidney folder
annoy_path = Path(str(ref_path)).with_suffix(".idx.annoy")
if not annoy_path.exists():
    # fallback to imported annoy if present
    alt = Path("reference")
    if alt.exists():
        # look for annoy files in repo
        annoy_candidates = list(alt.parent.glob("**/*.annoy"))
        if len(annoy_candidates) > 0:
            annoy_path = annoy_candidates[0]

celltype_key = "annotation.l2"  # change if your reference uses a different key
# default TAL labels (case-insensitive substring matching will be used)
tal_labels = ["Thick Ascending Limb"]  # add alternative labels if present

print("ref_path:", ref_path)
print("annoy_path:", annoy_path)
print("celltype_key:", celltype_key)
print("tal_labels:", tal_labels)

ref_path: reference/ref_kpmp.h5ad
annoy_path: reference/ref_kpmp.idx.annoy
celltype_key: annotation.l2
tal_labels: ['Thick Ascending Limb']


In [ ]:
# Prefer ref.h5ad if it exists, else keep the selected ref_path
from pathlib import Path
p2 = Path('reference/ref.h5ad')
if p2.exists():
    ref_path = p2
    print('Overriding ref_path to', ref_path)
else:
    print('No reference/ref.h5ad found; using', ref_path)

# Ensure annoy path matches
annoy_path = ref_path.with_suffix('.idx.annoy')
if not annoy_path.exists():
    # Fallback to looking in parent dir if not found alongside h5ad
    alt_annoy = Path('idx.annoy')
    if alt_annoy.exists():
        annoy_path = alt_annoy
print('Using annoy_path:', annoy_path)

Overriding ref_path to reference/ref_lake.h5ad


In [5]:
# Load the H5AD (or show a helpful error)
if not ref_path.exists():
    raise FileNotFoundError(f"Ref .h5ad not found at {ref_path}. If you only have ref.Rds, run the R conversion:\nRscript convert_ref_rds_to_h5ad.R path/to/ref.Rds {ref_path}")

print('Loading', ref_path)
if sc is not None:
    adata = sc.read_h5ad(str(ref_path))
else:
    import anndata as _ad
    adata = _ad.read_h5ad(str(ref_path))

print('Loaded AnnData with shape:', adata.shape)
print(adata)

Loading reference/ref_lake.h5ad
Loaded AnnData with shape: (64693, 29732)
AnnData object with n_obs × n_vars = 64693 × 29732
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'library', 'annotation.l1', 'annotation.l2', 'annotation.l3', 'subclass.full', 'celltype'
    layers: 'counts'


In [6]:
# Show basic obs/var info and warn if no annotation keys present
print('obs columns:', list(adata.obs.columns))
print('var columns:', list(adata.var.columns)[:10])
candidate_keys = [k for k in ['annotation.l1','annotation.l2','annotation.l3','celltype','cell_type'] if k in adata.obs.columns]
if len(candidate_keys) == 0:
    print('Warning: No obvious cell-type annotation keys found in adata.obs; TAL detection may fail.')
else:
    print('Candidate cell-type keys found:', candidate_keys)

obs columns: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'library', 'annotation.l1', 'annotation.l2', 'annotation.l3', 'subclass.full', 'celltype']
var columns: []
Candidate cell-type keys found: ['annotation.l1', 'annotation.l2', 'annotation.l3', 'celltype']


In [7]:
import numpy as np, scipy.sparse as sp

print("shape:", adata.shape)
print("layers:", list(getattr(adata, "layers", {}).keys()))
print("raw present:", getattr(adata, "raw", None) is not None)
print("adata.X: sparse:", sp.issparse(adata.X), "dtype:", getattr(adata.X, "dtype", None))
if sp.issparse(adata.X):
    print("X nnz:", adata.X.nnz, "sum:", adata.X.sum())
else:
    print("X non-zero count:", np.count_nonzero(adata.X), "sum:", adata.X.sum())
# top genes by sum
sums = (adata.X.sum(axis=0)).A1 if sp.issparse(adata.X) else adata.X.sum(axis=0)
top_idx = np.argsort(-sums)[:20]
print("Top gene sums (first 20):", sums[top_idx])
# check obs count-like columns
for c in adata.obs.columns:
    if 'ncount' in c.lower() or c.lower().startswith('ncount_') or 'total_counts' in c.lower():
        print(c, adata.obs[c].describe())

shape: (64693, 29732)
layers: ['counts']
raw present: False
adata.X: sparse: True dtype: int64
X nnz: 128414684 sum: 274131711
Top gene sums (first 20): [20390645   917674   912842   869465   843134   772213   591950   561784
   554708   526478   494162   441519   436429   435797   430627   405335
   404654   394247   392686   377827]
nCount_RNA count    64693.000000
mean      4237.424621
std       3406.117201
min        455.000000
25%       1677.000000
50%       3443.000000
75%       5752.000000
max      38089.000000
Name: nCount_RNA, dtype: float64


In [8]:
# Try to load schema JSON and sample CSVs written by the R converter (if present)
from pathlib import Path
import json

schema_path = Path(str(ref_path)).with_suffix('.schema.json')
obs_sample_path = Path(str(ref_path)).with_suffix('.obs_sample.csv')
var_sample_path = Path(str(ref_path)).with_suffix('.var_sample.csv')

if schema_path.exists():
    print('Found schema summary:', schema_path)
    with open(schema_path) as f:
        schema = json.load(f)
    # Pretty-print a few interesting fields
    print('class:', schema.get('class'))
    print('n_cells:', schema.get('n_cells'))
    print('n_genes:', schema.get('n_genes'))
    print('assays:', [a['name'] for a in schema.get('assays', [])])
    print('reductions:', [r['name'] for r in schema.get('reductions', [])])
    if 'possible_celltype_keys' in schema:
        print('candidate celltype keys:', schema.get('possible_celltype_keys'))
else:
    print('No schema JSON found at', schema_path)

if obs_sample_path.exists():
    print('\nShowing head of obs sample:')
    display(pd.read_csv(obs_sample_path).head())
else:
    print('No obs sample CSV found at', obs_sample_path)

if var_sample_path.exists():
    print('\nShowing head of var sample:')
    display(pd.read_csv(var_sample_path).head())
else:
    print('No var sample CSV found at', var_sample_path)


No schema JSON found at reference/ref_lake.schema.json
No obs sample CSV found at reference/ref_lake.obs_sample.csv
No var sample CSV found at reference/ref_lake.var_sample.csv


In [9]:
# Load exported reductions and cell-type counts (if present)
from pathlib import Path
base = Path(str(ref_path)).with_suffix('')

# reductions
reds_glob = list(base.parent.glob(base.name + '.*.csv'))
reduction_files = [p for p in reds_glob if not p.name.endswith('.obs_sample.csv') and not p.name.endswith('.var_sample.csv') and not p.name.endswith('.counts.csv')]
if len(reduction_files) > 0:
    print('Found reduction files:')
    for p in reduction_files:
        print(' -', p)
    # Prefer refUMAP if present
    umap_file = [p for p in reduction_files if 'refUMAP' in p.name or 'umap' in p.name.lower()]
    if len(umap_file) > 0:
        umap_file = umap_file[0]
        print('\nLoading UMAP from', umap_file)
        df_umap = pd.read_csv(umap_file, index_col=0)
        display(df_umap.head())
        # quick scatter
        plt.figure(figsize=(6,6))
        plt.scatter(df_umap.iloc[:,0], df_umap.iloc[:,1], s=4, c='lightgray')
        plt.title('Reference embedding (from file)')
        plt.axis('off')
        plt.show()
    else:
        print('\nNo UMAP-specific file found among reductions; you can inspect the available reduction CSVs listed above')
else:
    print('No reduction CSVs found next to h5ad')

# cell-type counts
count_files = list(base.parent.glob(base.name + '.*.counts.csv'))
if len(count_files) > 0:
    print('\nFound count files:')
    for p in count_files:
        print(' -', p)
        # Read CSV robustly: headers may be present (R wrote a header), so let pandas infer header
        dfc = pd.read_csv(p, index_col=0)
        # Coerce to a single numeric 'count' column if necessary
        if dfc.shape[1] == 1:
            dfc.columns = ['count']
        else:
            # prefer first numeric column
            num_col = None
            for col in dfc.columns:
                if np.issubdtype(dfc[col].dtype, np.number):
                    num_col = col
                    break
            if num_col is None:
                # try converting the first column to numeric
                try:
                    dfc.iloc[:, 0] = pd.to_numeric(dfc.iloc[:, 0])
                except Exception:
                    pass
                num_col = dfc.columns[0]
            dfc = dfc[[num_col]].copy()
            dfc.columns = ['count']
        display(dfc.head(20))
else:
    print('No count CSVs found')

No reduction CSVs found next to h5ad
No count CSVs found


In [10]:
# Inspect AnnData components
print('obs cols:', list(adata.obs.columns)[:30])
print('\nvar cols:', list(adata.var.columns)[:30])
print('\nobsm keys:', list(adata.obsm.keys()))

# Show small previews
print('\nadata.obs.head():')
display(adata.obs.head())

print('\nadata.var.head():')
display(adata.var.head())

# optionally show a small slice of the expression matrix
print('\nadata.X shape:', getattr(adata, 'X').shape)
if scipy.sparse.issparse(adata.X):
    sp_frac = (adata.X.nnz / float(adata.X.shape[0] * adata.X.shape[1]))
    print(f'sparsity fraction: {1-sp_frac:.3f} (dense fraction ~ {sp_frac:.3f})')


obs cols: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'library', 'annotation.l1', 'annotation.l2', 'annotation.l3', 'subclass.full', 'celltype']

var cols: []

obsm keys: []

adata.obs.head():


,orig.ident,nCount_RNA,nFeature_RNA,library,annotation.l1,annotation.l2,annotation.l3,subclass.full,celltype
Cell_1,Cell,531,441,KC33,IMM,MAC-R,MAC-R,Tissue-resident Macrophage,MAC-R
Cell_2,Cell,2298,1415,KC33,DCT,DCT,DCT1,Distal Convoluted Tubule Cell Type 1,DCT
Cell_3,Cell,677,528,KC33,PT,PT-S1,PT-S1,Proximal Tubule Epithelial Cell Segment 1,PT-S1
Cell_4,Cell,2230,1430,KC33,PT,PT-S1,PT-S1,Proximal Tubule Epithelial Cell Segment 1,PT-S1
Cell_5,Cell,3132,1830,KC33,PT,PT-S1,PT-S1,Proximal Tubule Epithelial Cell Segment 1,PT-S1



adata.var.head():


""
gene
Feature1
Feature2
Feature3
Feature4
Feature5



adata.X shape: (64693, 29732)
sparsity fraction: 0.933 (dense fraction ~ 0.067)


In [ ]:
# Find embedding and plot colored by cell type (or highlight TAL)
emb = None
try:
    emb = find_embedding(adata)
    print('Embedding shape:', emb.shape)
except Exception as e:
    print('Embedding detection failed:', e)
    # Fallback: try to compute a lightweight 2D embedding (PCA -> UMAP or sklearn fallbacks)
    try:
        print('Attempting to compute a fallback 2D embedding (PCA -> UMAP/TSNE)')
        # Choose an expression matrix (prefer raw if present)
        X_mat = None
        if getattr(adata, 'raw', None) is not None and getattr(adata.raw, 'X', None) is not None:
            X_mat = adata.raw.X
        else:
            X_mat = adata.X

        # Use scanpy if available (PCA+UMAP)
        if sc is not None:
            at = adata.copy()
            try:
                sc.pp.normalize_total(at, target_sum=1e4)
                sc.pp.log1p(at)
            except Exception:
                pass
            try:
                sc.pp.pca(at, n_comps=20)
                sc.pp.neighbors(at, use_rep='X_pca')
                sc.tl.umap(at)
                emb = at.obsm.get('X_umap')
                if emb is not None:
                    emb = emb[:, :2]
            except Exception as e2:
                print('scanpy-based embedding failed:', e2)
                emb = None

        # sklearn / umap fallback
        if emb is None:
            # compute PCA/TruncatedSVD -> UMAP/TSNE
            try:
                if scipy.sparse.issparse(X_mat):
                    from sklearn.decomposition import TruncatedSVD
                    svd = TruncatedSVD(n_components=20, random_state=SEED)
                    pcs = svd.fit_transform(X_mat)
                else:
                    from sklearn.decomposition import PCA
                    pcs = PCA(n_components=20, random_state=SEED).fit_transform(X_mat)
                try:
                    import umap as _umap
                    emb = _umap.UMAP(n_components=2, random_state=SEED).fit_transform(pcs)
                except Exception:
                    from sklearn.manifold import TSNE
                    emb = TSNE(n_components=2, random_state=SEED).fit_transform(pcs)
                print('Computed fallback embedding; shape:', emb.shape)
            except Exception as e3:
                print('Fallback embedding computation failed:', e3)
                emb = None
    except Exception as e4:
        print('Could not compute fallback embedding:', e4)

# Plot coloring by celltype_key if present
if celltype_key in adata.obs.columns:
    cats = adata.obs[celltype_key].astype('category')
    ncats = len(cats.cat.categories)
    print(f'Found {ncats} categories in {celltype_key}')

    if emb is None:
        print('No 2D embedding available; skipping embedding plots.')
    else:
        plt.figure(figsize=(6,6))
        if ncats <= 10:
            # color by categories
            colors = {c: plt.cm.tab10(i % 10) for i, c in enumerate(cats.cat.categories)}
            for c in cats.cat.categories:
                mask = (adata.obs[celltype_key] == c).to_numpy()
                plt.scatter(emb[~mask,0], emb[~mask,1], c='lightgray', s=6, alpha=0.4)
                plt.scatter(emb[mask,0], emb[mask,1], c=[colors[c]], s=8, label=str(c), alpha=0.8)
            plt.legend(fontsize='small', bbox_to_anchor=(1.05, 1))
            plt.title('Embedding colored by '+celltype_key)
            plt.axis('off')
        else:
            # highlight TAL instead of full legend (case-insensitive substring match)
            import re
            pattern = '|'.join([re.escape(str(x)) for x in tal_labels]) if len(tal_labels) > 0 else ''
            if pattern == '':
                tal_mask = pd.Series(False, index=adata.obs.index)
                print('No tal_labels provided; nothing highlighted.')
            else:
                tal_mask = adata.obs[celltype_key].astype(str).str.contains(pattern, case=False, na=False)
                if tal_mask.sum() == 0:
                    print('No TAL labels matched for the current `tal_labels` and `celltype_key`; attempting marker-based detection...')
                    # Marker-based fallback
                    markers = ['UMOD', 'SLC12A1']
                    present = [m for m in markers if m in adata.var_names]
                    if len(present) == 0:
                        print('No canonical TAL markers found in var_names:', markers)
                    else:
                        print('Markers present:', present)
                        # compute simple marker-positive masks
                        def _expr_vec(g):
                            col = adata[:, g].X
                            if scipy.sparse.issparse(col):
                                return np.asarray(col.toarray()).reshape(-1)
                            else:
                                return np.asarray(col).reshape(-1)
                        exprs = [_expr_vec(g) for g in present]
                        # require all markers >0
                        marker_mask = np.ones(adata.n_obs, dtype=bool)
                        for e in exprs:
                            marker_mask &= (e > 0)
                        if marker_mask.sum() == 0:
                            # relaxed: any marker >0
                            any_mask = np.zeros(adata.n_obs, dtype=bool)
                            for e in exprs:
                                any_mask |= (e > 0)
                            if any_mask.sum() > 0:
                                tal_mask = any_mask
                                print('Relaxed marker heuristic found', tal_mask.sum(), 'candidate TAL cells')
                            else:
                                print('Marker heuristics found no TAL candidates')
                                tal_mask = pd.Series(False, index=adata.obs.index)
                        else:
                            tal_mask = pd.Series(marker_mask, index=adata.obs.index)
                            print('Marker heuristic found', tal_mask.sum(), 'candidate TAL cells')
            # Only highlight if we found some TALs
            if tal_mask.sum() == 0:
                print('No TAL cells found with current settings; nothing highlighted.')
                plt.scatter(emb[:,0], emb[:,1], c='lightgray', s=6, alpha=0.5)
            else:
                plt.scatter(emb[~tal_mask.values,0], emb[~tal_mask.values,1], c='lightgray', s=6, alpha=0.5)
                plt.scatter(emb[tal_mask.values,0], emb[tal_mask.values,1], c='red', s=10, alpha=0.9)
            plt.title('Embedding: TAL highlighted in red')
            plt.axis('off')
        plt.show()
else:
    print(f"Column {celltype_key} not found in adata.obs. Available keys: {list(adata.obs.columns)[:10]}")


In [ ]:
import numpy as np
import scipy.sparse as sp

print("obs columns:", list(adata.obs.columns)[:50])
for c in ['nCount_RNA','n_counts','total_counts','library_size']:
    if c in adata.obs.columns:
        print(c, adata.obs[c].describe())

print("layers:", list(adata.layers.keys()))
for l in adata.layers.keys():
    arr = adata.layers[l]
    print(f"layer {l}: shape={arr.shape}, sparse={sp.issparse(arr)}")
    if sp.issparse(arr):
        print("  nnz:", arr.nnz, "sample row sums:", np.asarray(arr.sum(axis=1)).reshape(-1)[:10])
    else:
        print("  sample row sums:", np.sum(arr, axis=1)[:10])

print("adata.X: shape=", adata.X.shape, "sparse=", sp.issparse(adata.X), "dtype=", getattr(adata.X, 'dtype', None))
if sp.issparse(adata.X):
    print("  nnz:", adata.X.nnz, "row sums sample:", np.asarray(adata.X.sum(axis=1)).reshape(-1)[:10])
else:
    print("  row sums sample:", np.sum(adata.X, axis=1)[:10])

if getattr(adata, 'raw', None) is not None:
    rawX = adata.raw.X
    print("raw.X: shape=", rawX.shape, "sparse=", sp.issparse(rawX))
    if sp.issparse(rawX):
        print("  raw nnz:", rawX.nnz, "raw row sums sample:", np.asarray(rawX.sum(axis=1)).reshape(-1)[:10])
    else:
        print("  raw row sums sample:", np.sum(rawX, axis=1)[:10])
else:
    print("adata.raw is None")

In [ ]:
# UMI summary and histogram
umis = compute_umis(adata)
print('UMI stats (min, median, mean, max):', np.min(umis), np.median(umis), np.mean(umis), np.max(umis))

plt.figure(figsize=(6,3))
plt.hist(umis, bins=50, color='C0', alpha=0.8)
plt.xlabel('UMI counts'); plt.ylabel('Cells'); plt.title('UMI distribution')
plt.tight_layout(); plt.show()


In [ ]:
# Diagnostic: list labels across candidate cell-type keys and suggest TAL labels
import re
candidate_keys = [k for k in ['annotation.l1','annotation.l2','annotation.l3'] if k in adata.obs.columns]
print('Candidate cell-type keys found:', candidate_keys)

# detection function focused on Thick Ascending Limb (TAL)
def is_tal_label(s: str) -> bool:
    if s is None:
        return False
    s = str(s).lower()
    # prefer labels that explicitly contain both 'thick' and 'ascending'
    if 'thick' in s and 'ascending' in s:
        return True
    # or exact phrase 'thick ascending limb'
    if 'thick ascending' in s:
        return True
    # also accept 'cortical thick ascending limb' / 'medullary ...'
    if re.search(r'\bcortical .*thick.*ascending\b', s) or re.search(r'\bmedullary .*thick.*ascending\b', s):
        return True
    return False

# collect suggestions per key
tal_suggestions = []
for k in candidate_keys:
    print(f"\nKey: {k} ({adata.obs[k].nunique()} categories)")
    vc = adata.obs[k].value_counts()
    display(vc.head(50))
    matches = [str(c) for c in vc.index if is_tal_label(c)]
    if matches:
        print('  => Potential TAL labels (detected):', matches)
        tal_suggestions.extend([(k, m) for m in matches])

if tal_suggestions:
    print('\nSuggested TAL labels (key, label):')
    for k, m in tal_suggestions:
        print(f" - ({k}) {m}")
    # choose the first key with matches as the celltype_key by default
    chosen_key = tal_suggestions[0][0]
    proposed = list(dict.fromkeys([m for (_, m) in tal_suggestions]))
    print('\nAuto-selecting celltype_key =', chosen_key)
    print('Auto-setting tal_labels =', proposed)
    # only change variables if user left the defaults unchanged (helpful automation)
    # If user already set tal_labels to something that yielded matches, do not override
    current_mask = adata.obs.get(celltype_key, pd.Series(dtype=bool)).astype(str).isin(tal_labels)
    if current_mask.sum() == 0:
        celltype_key = chosen_key
        tal_labels = proposed
        print('\nUpdated variables:')
        print('  celltype_key ->', celltype_key)
        print('  tal_labels ->', tal_labels)
    else:
        print('\nExisting tal_labels already match some cells; keeping user-specified values.')
else:
    print('\nNo explicit TAL labels detected (looking for labels containing both "thick" and "ascending").')
    print('You can try setting `tal_labels` manually (examples: "Cortical Thick Ascending Limb", "Medullary Thick Ascending Limb").')

# Now create a flexible selection mask (case-insensitive substring match) and proceed
pattern = '|'.join([re.escape(str(x)) for x in tal_labels])
mask = adata.obs[celltype_key].astype(str).str.contains(pattern, case=False, na=False)
print('\nCells matching tal_labels (count):', int(mask.sum()))

if mask.sum() == 0:
    print('No TAL cells found with current settings; review the candidate keys above and adjust `celltype_key` / `tal_labels` accordingly.')
else:
    print('Proceeding with TAL subset using current settings.')

In [ ]:
# Recompute TAL subset (flexible substring matching) and recompute gene stats if previous attempt failed or was strict
import re
pattern = '|'.join([re.escape(str(x)) for x in tal_labels]) if len(tal_labels) > 0 else ''
if pattern == '':
    print('No tal_labels defined; skipping TAL recompute.')
else:
    tal_mask = adata.obs[celltype_key].astype(str).str.contains(pattern, case=False, na=False)
    if tal_mask.sum() == 0:
        print('Recompute: no TAL matches found; please review `celltype_key` and `tal_labels`.')
    else:
        adata_tal = adata[tal_mask].copy()
        print('Recomputed TAL subset:', adata_tal.n_obs, 'cells')
        # recompute simple gene stats and save
        if scipy.sparse.issparse(adata_tal.X):
            is_expr = (adata_tal.X > 0)
            prevalence = np.asarray(is_expr.sum(axis=0)).reshape(-1) / float(adata_tal.n_obs)
            sums = np.asarray(adata_tal.X.sum(axis=0)).reshape(-1)
            means = sums / float(adata_tal.n_obs)
        else:
            x = np.asarray(adata_tal.X)
            prevalence = (x > 0).sum(axis=0) / float(adata_tal.n_obs)
            means = x.mean(axis=0)
        genes = np.asarray(adata_tal.var_names)
        df_stats = pd.DataFrame({'gene': genes, 'prevalence': prevalence, 'mean_expr': means})
        df_stats.sort_values('mean_expr', ascending=False, inplace=True)
        out_dir = Path('examples')
        out_dir.mkdir(exist_ok=True)
        df_stats.to_csv(out_dir / 'tal_gene_stats.csv', index=False)
        print('Updated examples/tal_gene_stats.csv with recomputed TAL stats')

In [ ]:
# Subset to TAL cells and compute simple gene stats for TAL
if celltype_key in adata.obs.columns:
    tal_mask = adata.obs[celltype_key].isin(tal_labels).to_numpy()
    n_tal = int(tal_mask.sum())
    if n_tal == 0:
        # helpful diagnostic: show top categories to guide user
        cats = adata.obs[celltype_key].astype('category')
        top_cats = list(cats.cat.categories)[:20]
        raise ValueError(
            f"No TAL cells found (0 matches for {tal_labels}) in column '{celltype_key}'. "
            f"Available categories (top): {top_cats[:10]}\n" 
            "Try adjusting `celltype_key` or `tal_labels` to match your reference."
        )
    adata_tal = adata[tal_mask].copy()
    print('TAL cells:', adata_tal.n_obs, '/', adata.n_obs, f'({adata_tal.n_obs/adata.n_obs:.2%})')
else:
    raise ValueError(f"Cell type key '{celltype_key}' not found in adata.obs")

# Compute prevalence and mean expression in TAL
if scipy.sparse.issparse(adata_tal.X):
    is_expr = (adata_tal.X > 0)
    prevalence = np.asarray(is_expr.sum(axis=0)).reshape(-1) / float(adata_tal.n_obs)
    # compute means safely as sum / n_obs to avoid scipy's sparse mean behavior when n_obs==0
    sums = np.asarray(adata_tal.X.sum(axis=0)).reshape(-1)
    means = sums / float(adata_tal.n_obs)
else:
    x = np.asarray(adata_tal.X)
    prevalence = (x > 0).sum(axis=0) / float(adata_tal.n_obs)
    means = x.mean(axis=0)

genes = np.asarray(adata_tal.var_names)

df_stats = pd.DataFrame({'gene': genes, 'prevalence': prevalence, 'mean_expr': means})
df_stats.sort_values('mean_expr', ascending=False, inplace=True)
print('\nTop 20 genes by mean expression in TAL:')
display(df_stats.head(20))

# Save small tables
out_dir = Path('examples')
out_dir.mkdir(exist_ok=True)
df_stats.to_csv(out_dir / 'tal_gene_stats.csv', index=False)
print('Saved examples/tal_gene_stats.csv')


In [ ]:
# Plot expression overlays for a small panel of genes
# choose controls (if not provided, use top mean genes excluding MT/RPS/RPL)
controls = []
if len(controls) == 0:
    import re
    bad_re = re.compile(r'^(MT-|mt-|RPS|RPL)')
    filtered = [g for g in df_stats['gene'].values if not bad_re.match(g)]
    panel = filtered[:6]
else:
    panel = [g for g in controls if g in genes][:6]

print('Plotting panel genes:', panel)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, gene in zip(axes.ravel(), panel):
    # get expression vector
    vec = adata_tal[:, gene].X
    if scipy.sparse.issparse(vec):
        vec = np.asarray(vec.toarray()).reshape(-1)
    else:
        vec = np.asarray(vec).reshape(-1)
    # FG as top 10%
    thr = np.quantile(vec, 0.9)
    if thr == 0:
        fg = vec > 0
    else:
        fg = vec >= thr

    sc_emb = find_embedding(adata_tal)
    ax.scatter(sc_emb[~fg,0], sc_emb[~fg,1], c='lightgray', s=6, alpha=0.4)
    ax.scatter(sc_emb[fg,0], sc_emb[fg,1], c='red', s=10, alpha=0.85)
    # color by expression for background for contrast
    sc = ax.scatter(sc_emb[:,0], sc_emb[:,1], c=vec, s=6, cmap='viridis', alpha=0.8)
    ax.set_title(gene)
    ax.axis('off')
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)

plt.tight_layout()
plt.show()


In [ ]:
# Optional: try loading an Annoy index if present and demonstrate a small query
if annoy_path.exists():
    try:
        from annoy import AnnoyIndex
        # Try to guess dimension from refDR or similar
        dim = 50 # default fallback
        if 'X_refDR' in adata.obsm:
             dim = adata.obsm['X_refDR'].shape[1]
        elif 'X_pca' in adata.obsm:
             dim = adata.obsm['X_pca'].shape[1]
        
        print(f"Assuming Annoy index dimension: {dim}")
        t = AnnoyIndex(dim, metric='angular')
        t.load(str(annoy_path))
        print('Loaded Annoy index from', annoy_path)
        # Query a random TAL cell
        if 'adata_tal' in locals():
            idx_sample = np.random.randint(0, adata_tal.n_obs)
            q_vec = find_embedding(adata_tal)[idx_sample]
            # Note: q_vec from find_embedding is 2D UMAP, but index expects refDR/PCA.
            # We can't query the index with UMAP coordinates if it was built on PCA.
            # We need the high-dim vector for the query.
            # But we don't have the high-dim vector for adata_tal easily unless we subset refDR.
            # Let's skip the query if we can't get the right vector, or just show we loaded it.
            print("Index loaded. (Skipping query demo as vector space might differ from visualization)")
        else:
             print("Index loaded.")
    except Exception as e:
        print('Could not use Annoy index:', e)
else:
    print('No Annoy index found at', annoy_path)

In [ ]:
# Reproducibility: package versions and seed
import json
session_info = {
    'python': sys.version.splitlines()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'matplotlib': mpl.__version__,
    'anndata': anndata.__version__ if anndata is not None else None,
    'scanpy': sc.__version__ if sc is not None else None,
    'seed': SEED
}
print(json.dumps(session_info, indent=2))
with open('examples/explore_ref_session_info.json', 'w') as f:
    json.dump(session_info, f, indent=2)
print('Wrote examples/explore_ref_session_info.json')

# Final notes

- This notebook is for exploratory analysis and quick diagnostics; it purposely avoids heavy compute (no permutation tests).
- Use `examples/case_study_1_tal.py` for a full, reproducible BioRSP run (this notebook helps inspect inputs and pick sensible parameters).
- If you want, I can add a small cell to compute FG neighbor fraction using Annoy and save it as a diagnostic CSV.

Enjoy — run the cells in order and edit `ref_path` at the top to point to your data.